Make sure the ___src___ folder is in system path for imports to work

In [17]:
import sys
sys.path.append(r'..\src')

In [19]:
import config
from data import load_data
config = config.ProjectConfig() 
print(config.random_state) # test config for the project so that everything is consistent

features, properties = load_data('../data/features.xlsx', '../data/properties.xlsx') # load data

22


### Tune the model hyperparameters

In [21]:
from tuning import tune_models
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# parameter grids for different sklearn regressors
# random forest
param_grid_rf = {'n_estimators': [100, 200, 250], 'max_depth': [5, 10, 15], 'min_samples_leaf':[5,7,10]}
# decision_tree
param_grid_dt = {'max_depth': [2, 5, 10], 'min_samples_leaf':[5,7,10], 'splitter':['best', 'random'],'min_samples_split':[10,20,30]}
# XGB
param_grid_xgb = {'n_estimators': [100, 200, 250], 'max_depth': [6, 10, 15], 'learning_rate':[0.01,0.05,0.1]}

dtreg = DecisionTreeRegressor()
rfreg = RandomForestRegressor()
xgbreg = XGBRegressor()

param_grid_list = [param_grid_rf, param_grid_dt, param_grid_xgb]
model_name_list = ['random_forest','decision_tree', 'XGB']#'KNN']
model_instance_list = [rfreg, dtreg, xgbreg]

# tune the hyperparameters using a grid search 
for estimator, param_grid, model_name in zip(model_instance_list, param_grid_list, model_name_list):
    print(f'---{model_name}---')
    tune_models(features,properties,estimator,param_grid,config,model_name)

---random_forest---
------------------------------
tuning for target property CO(conv)
------------------------------
CO(conv): best parameters = {'max_depth': 10, 'min_samples_leaf': 5, 'n_estimators': 200}
------------------------------
tuning for target property CH4
------------------------------
CH4: best parameters = {'max_depth': 10, 'min_samples_leaf': 5, 'n_estimators': 100}
------------------------------
tuning for target property CO2
------------------------------
CO2: best parameters = {'max_depth': 5, 'min_samples_leaf': 5, 'n_estimators': 250}
------------------------------
tuning for target property C2-C4
------------------------------
C2-C4: best parameters = {'max_depth': 15, 'min_samples_leaf': 5, 'n_estimators': 250}
------------------------------
tuning for target property С5+
------------------------------
С5+: best parameters = {'max_depth': 10, 'min_samples_leaf': 5, 'n_estimators': 250}
------------------------------
tuning for target property Mult
--------------

### Evaluate models with the best parameters

In [24]:
from evaluation import evaluate_models
from sklearn.base import clone
import json
from glob import glob
json_output_paths = glob(f"{config.output_dir}/*.json") # get paths for best_parameters json files
print(json_output_paths)
model_name_list = ['decision_tree', 'random_forest', 'XGB']#'KNN']
model_instance_list = [dtreg, rfreg, xgbreg]

for estimator, output_path, model_name in zip(model_instance_list, json_output_paths, model_name_list):
    with open(output_path, "r", encoding="utf-8") as file:
        best_params = json.load(file)
    evaluate_models(features,properties,estimator,best_params,config,model_name)

['artifacts\\decision_tree_best_parameters.json', 'artifacts\\random_forest_best_parameters.json', 'artifacts\\XGB_best_parameters.json']


### Plot model performance (MAE and R2)

In [25]:
from plotting import plot_performance
import pandas as pd
csv_output_paths = glob(f"{config.output_dir}/*.csv")
for csvs in csv_output_paths:
    results_df = pd.read_csv(csvs)
    plot_performance(results_df, config.plot_dir)

### SHAP and ICE analysis for the best performing model </br> 
As the XGB model was best performing (see plots in folder ___reports/figures___), I choose the XGB model for SHAP and ICE analysis. This could be different for different datasets

In [26]:
from sklearn.model_selection import train_test_split
from plot_shap import plot_shap
import pickle

model_name = 'XGB'

target_shap_values = {}

for target in properties.columns:
    
    X_train, X_test, y_train, y_test = train_test_split(
        features,properties[target],test_size=config.test_size,random_state=config.random_state)
    X_background = pd.DataFrame(X_train, columns=features.columns)
    
    # load the models from the .pkl files that was saved during the evaluation stage
    with open(f"{config.output_dir}/{model_name}_{target}_model.pkl", 'rb') as file:
        model = pickle.load(file)
        
    shap_values = plot_shap(model,X_background,target,config.plot_dir)
    
    target_shap_values[target] = shap_values